# Week 3 — Data Contract
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

> **Run this in Colab**, with `HF_TOKEN` stored as a Secret (never pasted in a cell).
> Cell 0 explores the real warehouse structure — run it first and adapt table/column
> names in later cells to what you actually see. Everywhere you see `# ADAPT:` is a
> spot to fill in from real output, not to take on faith.


## 0. Setup + explore the real schema

Authenticate, then list what's actually in the warehouse before assuming any table
or column name. This keeps the rest of the notebook honest instead of guessed.


In [ ]:
# Setup
!pip -q install huggingface_hub duckdb pyarrow pandas

import os
from huggingface_hub import login, HfApi

# Reads the Colab Secret named HF_TOKEN -- do NOT paste a token here
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])

api = HfApi()
REPO = "FlyRank/internship-warehouse"
files = api.list_repo_files(REPO, repo_type="dataset")
for f in sorted(files)[:60]:
    print(f)
print(f"\n...{len(files)} files total")


In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

# ADAPT: replace with the actual table path you saw printed above for a mid-panel month.
# Hive-partitioned tables usually look like: hf://datasets/FlyRank/internship-warehouse/<table>/month=2026-03/*.parquet
TABLE_PATH = "hf://datasets/FlyRank/internship-warehouse/<table_name>/month=2026-03/*.parquet"

schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{TABLE_PATH}')").df()
schema


## 1. The contract, in plain words

**One row means:** *ADAPT — e.g. "one content item's (page's) observed search + engagement signals,
for one client, on one day"* — fill in from what you confirmed in Section 0/2, not from assumption.

**Table(s) I'll use:** *ADAPT — name the actual table(s) from the file listing above, e.g. a
GSC-daily fact table joined to a content dimension table.* Sticking to what Lane 2 (refresh
scoring) needs — search-visibility signals and engagement signals at the content grain — not
pulling in tables the lane doesn't use.

**Time window:** a **mid-panel month, `month=2026-03`**, per the brief's own instruction — *not*
`_sample` (that's the sealed final month, June 2026, reserved as a held-out test window and never
used to develop label logic).

**What I'd predict or rank (label or proxy):** carried over from Week 2 — `is_declining_label`
(proxy: recent trend direction down), moving toward a true future-window label
(prior 90 days → decline over next 30 days) once I can build that from daily-grain data here.

**One thing I deliberately exclude:** any FlyRank product-computed score or action field
(e.g. `health_score`, `priority_score`, `action_type`), if present in this warehouse slice — using
one as a feature would make any "signal" I find circular, since it would just be re-deriving
FlyRank's existing rule rather than learning from raw observed behavior.


## 2. Prove three facts with three small queries (`month=2026-03`)

### 2a. Grain check — one row really is what I said it is


In [ ]:
# ADAPT: replace grain_cols with the actual key(s) that should be unique per row
# for your stated grain (e.g. content_id + date, or content_id alone for a monthly rollup)
grain_cols = ["content_id", "date"]  # ADAPT

q = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT ({", ".join(grain_cols)})) AS distinct_keys
FROM read_parquet('{TABLE_PATH}')
"""
con.execute(q).df()
# total_rows should equal distinct_keys if the stated grain holds.
# If it doesn't, the contract's "one row means..." line above needs to be corrected, not the query.


### 2b. Row count and date span for my slice

In [ ]:
q = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(date) AS min_date,
    MAX(date) AS max_date,
    COUNT(DISTINCT client_id) AS n_clients   -- ADAPT if client column has a different name
FROM read_parquet('{TABLE_PATH}')
"""
con.execute(q).df()


### 2c. Availability — filter with `IS TRUE`, show survivors

`IS TRUE` (rather than `= TRUE` or `= 1`) matters here because it correctly drops both `FALSE`
*and* `NULL` rows in one filter — a plain `= TRUE` in some engines silently keeps or drops NULLs
inconsistently. This is the availability check the lane guide warns is easy to get quietly wrong.


In [ ]:
# ADAPT: replace with the real boolean availability column you saw in the schema,
# e.g. is_indexed, is_available, has_gsc_data -- whatever marks a row as usable/observed
AVAILABILITY_COL = "is_available"  # ADAPT

q = f"""
SELECT
    COUNT(*) AS rows_before,
    COUNT(*) FILTER (WHERE {AVAILABILITY_COL} IS TRUE) AS rows_after
FROM read_parquet('{TABLE_PATH}')
"""
result = con.execute(q).df()
result["share_surviving"] = result["rows_after"] / result["rows_before"]
result


## 3. Five features, max

Built from the same `month=2026-03` slice. Each feature gets one line: *knowable at the decision
moment because...* — the test being: could I have computed this on the day a reviewer would
actually look at the page, without peeking into the future?


In [ ]:
# ADAPT column names to match the real schema from Section 0.
# Pull a feature frame at the content grain for the month.

feat_q = f"""
SELECT
    content_id,
    client_id,
    SUM(impressions) AS impressions_month,           -- ADAPT
    SUM(clicks) AS clicks_month,                      -- ADAPT
    AVG(avg_position) AS avg_position_month,          -- ADAPT
    MAX(days_since_last_update) AS days_since_update, -- ADAPT
    AVG(word_count) AS word_count                     -- ADAPT
FROM read_parquet('{TABLE_PATH}')
GROUP BY content_id, client_id
"""
features = con.execute(feat_q).df()
print(features.shape)
features.head()


**Feature 1 — `impressions_month`**: knowable at the decision moment because it's a running sum
of search impressions observed up through the current date — no future data needed.

**Feature 2 — `clicks_month`**: same reasoning — an observed count of clicks that already
happened, summed up to "now."

**Feature 3 — `avg_position_month`**: an average of the page's observed search ranking position
so far this month; entirely backward-looking.

**Feature 4 — `days_since_update`**: a simple calendar fact (today minus last edit date) — always
knowable, never depends on anything that hasn't happened yet.

**Feature 5 — `word_count`**: a static property of the current published content — knowable the
moment the page exists in its current form, independent of any future traffic outcome.


## 4. The trap: deliberate leakage, then remove it

Adding one column that is **derived from the label itself** (or from data that would only exist
*after* the outcome is known), training a quick score, watching it jump toward suspiciously
perfect — then deleting the column and keeping the honest number. This is the same leakage lesson
from notebook 02, now on real warehouse data.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# ADAPT: join in the label from wherever it lives in this warehouse slice
# (or recompute is_declining_label the same way as Week 2, at this grain)
# labeled = features.merge(label_df, on=["content_id", "client_id"])
labeled = features.copy()
labeled["is_declining_label"] = 0  # ADAPT: replace with the real joined/derived label

honest_feature_cols = ["impressions_month", "clicks_month", "avg_position_month",
                        "days_since_update", "word_count"]

X = labeled[honest_feature_cols].fillna(0)
y = labeled["is_declining_label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Honest ROC AUC (5 real features only): {honest_auc:.3f}")


In [ ]:
# THE TRAP: add a label-derived column on purpose.
# e.g. trend_pct or trend_direction, which is literally how the label was computed --
# or any post-outcome signal that wouldn't exist at decision time.
leaky = labeled.copy()
leaky["leaky_trend_pct"] = 0  # ADAPT: pull in the real trend_pct / label-adjacent column here

leak_feature_cols = honest_feature_cols + ["leaky_trend_pct"]
X_leak = leaky[leak_feature_cols].fillna(0)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.25, random_state=42)
model_leak = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_auc = roc_auc_score(y_test_l, model_leak.predict_proba(X_test_l)[:, 1])

print(f"Honest ROC AUC: {honest_auc:.3f}")
print(f"Leaky ROC AUC:  {leaky_auc:.3f}  <-- jumps toward ~1.0, which is the tell, not a win")


**What happened:** adding `leaky_trend_pct` — a column derived directly from the same signal
that defines the label — let the model essentially look up the answer instead of predicting it.
The jump toward a near-perfect score is the leakage symptom, not genuine performance.

**Deleting it and keeping the honest number:** the real result for this week is the **honest ROC
AUC computed with the 5 real features only**, printed above — that's the number that survives into
next week, not the leaky one.


## Named limitation of this slice

*ADAPT once you've actually queried: e.g. "this month's slice only covers N clients, so any
pattern I see could be dominated by one or two large clients' behavior rather than being general,"
or "the availability filter drops X% of rows, so my feature frame is systematically missing pages
that were never indexed — the model will never learn about that population at all."* Name the real
one you observe, not a generic disclaimer.


## 5. Self-check

- [ ] Five plain-words contract answers filled in with real table/column names (Section 1)
- [ ] Exactly three verification queries run with real, visible output: grain, row count/date
      span, and availability filtered with `IS TRUE` (Section 2)
- [ ] Five-feature frame built from `month=2026-03`, each with an honest
      "available when?" line (Section 3)
- [ ] Deliberate-leak experiment shown (score jumps), then leaky column removed and the honest
      number kept as the real result (Section 4)
- [ ] One *specific, real* named limitation of this slice, not a generic disclaimer
- [ ] Ran this in Colab with `HF_TOKEN` as a Secret (never pasted directly in a cell)
